# 03.3 Continuous Batching: Static vs Dynamic Scheduling

Compare static (padded) batching against continuous (in-flight) batching.
Metrics: throughput (tokens/sec), GPU utilization over time, per-request completion timeline.

In [ ]:
import sys
sys.path.insert(0, '../../..')

import numpy as np
import matplotlib.pyplot as plt
from dataclasses import dataclass, field
from typing import List
import time

In [ ]:
@dataclass
class Request:
    id: int
    prompt_len: int
    gen_len: int
    arrive_step: int
    start_step: int = -1
    end_step: int = -1

def generate_requests(n=32, seed=42):
    rng = np.random.default_rng(seed)
    reqs = []
    for i in range(n):
        reqs.append(Request(
            id=i,
            prompt_len=rng.integers(32, 256),
            gen_len=rng.integers(16, 512),
            arrive_step=int(rng.exponential(3))
        ))
    return sorted(reqs, key=lambda r: r.arrive_step)

requests = generate_requests()
print(f"{len(requests)} requests, gen_len range: {min(r.gen_len for r in requests)}-{max(r.gen_len for r in requests)}")

In [ ]:
def simulate_static_batching(requests: List[Request], max_batch=8):
    """Static batching: wait for batch to fill, pad all to max gen_len in batch."""
    reqs = [Request(**r.__dict__) for r in requests]
    queue, step, completed = list(reqs), 0, []
    utilization = []  # (step, active_slots / max_batch)

    while queue or completed != reqs:
        # Grab up to max_batch from queue that have arrived
        available = [r for r in queue if r.arrive_step <= step]
        if not available:
            step += 1
            utilization.append((step, 0.0))
            continue

        batch = available[:max_batch]
        for r in batch:
            queue.remove(r)
            r.start_step = step

        max_gen = max(r.gen_len for r in batch)
        # All slots occupied for max_gen steps (padded)
        for t in range(max_gen):
            active = sum(1 for r in batch if (step + t - r.start_step) < r.gen_len)
            utilization.append((step + t, active / max_batch))

        for r in batch:
            r.end_step = step + max_gen  # all wait for longest
        completed.extend(batch)
        step += max_gen

        if len(completed) == len(reqs):
            break

    return completed, utilization

In [ ]:
def simulate_continuous_batching(requests: List[Request], max_batch=8):
    """Continuous batching: evict finished requests, admit new ones each step."""
    reqs = [Request(**r.__dict__) for r in requests]
    queue = list(reqs)
    active_batch: List[tuple] = []  # (request, tokens_generated)
    completed = []
    utilization = []
    step = 0

    while queue or active_batch:
        # Evict completed
        still_active = []
        for r, toks in active_batch:
            if toks >= r.gen_len:
                r.end_step = step
                completed.append(r)
            else:
                still_active.append((r, toks))
        active_batch = still_active

        # Admit new requests to fill slots
        available = [r for r in queue if r.arrive_step <= step]
        while len(active_batch) < max_batch and available:
            r = available.pop(0)
            queue.remove(r)
            r.start_step = step
            active_batch.append((r, 0))

        utilization.append((step, len(active_batch) / max_batch))

        # Generate one token per active request
        active_batch = [(r, toks + 1) for r, toks in active_batch]
        step += 1

        if not queue and not active_batch:
            break

    return completed, utilization

In [ ]:
static_completed, static_util = simulate_static_batching(requests)
cont_completed, cont_util = simulate_continuous_batching(requests)

def compute_throughput(completed):
    total_tokens = sum(r.gen_len for r in completed)
    total_steps = max(r.end_step for r in completed)
    return total_tokens / total_steps

static_tp = compute_throughput(static_completed)
cont_tp = compute_throughput(cont_completed)

print(f"Static batching:     {static_tp:.1f} tokens/step")
print(f"Continuous batching: {cont_tp:.1f} tokens/step")
print(f"Speedup: {cont_tp/static_tp:.2f}x")

In [ ]:
# Throughput bar chart
fig, ax = plt.subplots(figsize=(6, 4))
bars = ax.bar(['Static\n(Padded)', 'Continuous\n(In-flight)'], [static_tp, cont_tp],
              color=['#ef4444', '#22c55e'], edgecolor='black', width=0.5)
ax.set_ylabel('Tokens / Step')
ax.set_title('Throughput: Static vs Continuous Batching')
for bar in bars:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
            f'{bar.get_height():.1f}', ha='center', fontsize=11)
ax.set_ylim(0, max(static_tp, cont_tp) * 1.2)
plt.tight_layout()
plt.show()

In [ ]:
# GPU utilization over time
fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=False)

s_steps, s_utils = zip(*static_util)
axes[0].fill_between(s_steps, s_utils, alpha=0.4, color='#ef4444')
axes[0].plot(s_steps, s_utils, color='#ef4444', lw=1.5)
axes[0].set_ylabel('GPU Slot Utilization')
axes[0].set_title(f'Static Batching (avg util: {np.mean(s_utils):.1%})')
axes[0].set_ylim(0, 1.05)
axes[0].axhline(1.0, ls='--', color='gray', lw=0.8)

c_steps, c_utils = zip(*cont_util)
axes[1].fill_between(c_steps, c_utils, alpha=0.4, color='#22c55e')
axes[1].plot(c_steps, c_utils, color='#22c55e', lw=1.5)
axes[1].set_ylabel('GPU Slot Utilization')
axes[1].set_xlabel('Step')
axes[1].set_title(f'Continuous Batching (avg util: {np.mean(c_utils):.1%})')
axes[1].set_ylim(0, 1.05)
axes[1].axhline(1.0, ls='--', color='gray', lw=0.8)

plt.tight_layout()
plt.show()

In [ ]:
# Request completion timeline (Gantt-style)
fig, axes = plt.subplots(1, 2, figsize=(14, 6), sharey=True)

for ax, completed, title, color in [
    (axes[0], sorted(static_completed, key=lambda r: r.id), 'Static', '#ef4444'),
    (axes[1], sorted(cont_completed, key=lambda r: r.id), 'Continuous', '#22c55e')
]:
    for r in completed:
        ax.barh(r.id, r.end_step - r.start_step, left=r.start_step,
                color=color, alpha=0.7, edgecolor='black', linewidth=0.3)
    ax.set_xlabel('Step')
    ax.set_title(f'{title} Batching')
    ax.set_xlim(0, max(r.end_step for r in completed) * 1.05)

axes[0].set_ylabel('Request ID')
fig.suptitle('Request Completion Timeline', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Latency distribution comparison
static_latencies = [r.end_step - r.arrive_step for r in static_completed]
cont_latencies = [r.end_step - r.arrive_step for r in cont_completed]

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(static_latencies, bins=20, alpha=0.6, color='#ef4444', label=f'Static (p50={np.median(static_latencies):.0f})')
ax.hist(cont_latencies, bins=20, alpha=0.6, color='#22c55e', label=f'Continuous (p50={np.median(cont_latencies):.0f})')
ax.set_xlabel('End-to-End Latency (steps)')
ax.set_ylabel('Count')
ax.set_title('Per-Request Latency Distribution')
ax.legend()
plt.tight_layout()
plt.show()

## Key Takeaways

| Metric | Static | Continuous |
|--------|--------|------------|
| Throughput | Lower (padding waste) | Higher (slots always filled) |
| GPU Utilization | Drops as short requests finish | Near-100% sustained |
| Tail Latency | High (short reqs wait for long) | Lower (evict on completion) |

Continuous batching (Orca, vLLM, TGI) eliminates padding waste by treating the batch as a **dynamic set**:
- Finished sequences are evicted immediately
- New requests are admitted mid-generation
- Result: near-optimal GPU utilization regardless of sequence length variance